In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


## **Data Reading**

In [0]:
df=spark.read.format('parquet')\
    .load('abfss://bronze@projecte2e.dfs.core.windows.net/products')

In [0]:
df.display()

In [0]:
df=df.drop("_rescued_data")
df.display()

# **FUNCTION_using SQLudf**

In [0]:
df.createOrReplaceTempView("products")

In [0]:
%sql
create or replace function project_cata.bronze.discount_func(p_price double)
returns double
language SQL
return p_price*0.90

In [0]:
%sql
select product_id,price,project_cata.bronze.discount_func(price) as discounted_price from products

# Using Python

In [0]:
df=df.withColumn("discounted_price",expr("project_cata.bronze.discount_func(price)"))
df.display()

In [0]:
df.write.mode('overwrite')\
    .format('delta')\
        .option('path',"abfss://silver@projecte2e.dfs.core.windows.net/products")\
            .save()

In [0]:
%sql
create table if not exists project_cata.silver.products_silver
using delta
location "abfss://silver@projecte2e.dfs.core.windows.net/products"

In [0]:
%sql
select * from project_cata.silver.products_silver